In [14]:
!pip install -U -q bitsandbytes peft qwen-vl-utils

In [ ]:
!pip install -q git+https://github.com/huggingface/trl.git trackio

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
# !pip install flash-attn --no-build-isolation --extra-index-url https://download.pytorch.org/whl/cu121

In [ ]:
system_message = """You are a Vision Language Model specialized in interpreting visual data from chart images.
Your task is to analyze the provided chart image and respond to queries with concise answers, usually a single word, number, or short phrase.
The charts include a variety of types (e.g., line charts, bar charts) and contain colors, labels, and text.
Focus on delivering accurate, succinct answers based on the visual information. Avoid additional explanation unless absolutely necessary."""

def format_data(sample):
    return {
      "images": [sample["image"]],
      "messages": [
        {
            "role": "system",
            "content": [{"type": "text", "text": system_message}],
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "image"
                },
                {
                    "type": "text",
                    "text": sample['query']
                }
            ],
        },
        {
            "role": "assistant",
            "content": [{"type": "text", "text": sample["label"][0]}],
        },
      ]
    }

In [ ]:
import torch
from transformers import Idefics3ForConditionalGeneration, AutoProcessor

model_id = "HuggingFaceTB/SmolVLM-Instruct"

In [ ]:
from datasets import load_dataset

dataset_id = "HuggingFaceM4/ChartQA"
train_dataset, eval_dataset, test_dataset = load_dataset(dataset_id, split=['train[:10%]', 'val[:10%]', 'test[:10%]'])

train_dataset = [format_data(sample) for sample in train_dataset]
eval_dataset = [format_data(sample) for sample in eval_dataset]
test_dataset = [format_data(sample) for sample in test_dataset]

train_dataset[200]
print(len(train_dataset))

In [ ]:
train_dataset[1]

In [ ]:
train_dataset[0]['messages'][1:2]

In [ ]:
train_dataset[0]['images'][0]

In [ ]:
def generate_text_from_sample(model, processor, sample, max_new_tokens=1024, device="cuda"):
    # Prepare the text input by applying the chat template
    text_input = processor.apply_chat_template(
        sample['messages'][1:2],  # Use the sample without the system message
        add_generation_prompt=True
    )

    image_inputs = []
    image = sample['images'][0]
    #image = sample[1]['content'][0]['image']
    if image.mode != 'RGB':
        image = image.convert('RGB')
    image_inputs.append([image])

    # Prepare the inputs for the model
    model_inputs = processor(
        #text=[text_input],
        text=text_input,
        images=image_inputs,
        return_tensors="pt",
    ).to(device)  # Move inputs to the specified device

    # Generate text with the model
    generated_ids = model.generate(**model_inputs, max_new_tokens=max_new_tokens)

    # Trim the generated ids to remove the input ids
    trimmed_generated_ids = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    # Decode the output text
    output_text = processor.batch_decode(
        trimmed_generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )

    return output_text[0]  # Return the first decoded output text

def generate_text_from_sample_selector(model, processor, sample, max_new_tokens=20, device="cuda"):
    # 1. Pass the FULL message list (System + User), but exclude the Assistant's target answer
    # Assuming sample['messages'] is [System, User, Assistant]
    messages_to_prompt = sample['messages'][:2]

    text_input = processor.apply_chat_template(
        messages_to_prompt,
        add_generation_prompt=True # Ensures the <|im_start|>assistant\n tag is appended
    )

    image_inputs = []
    image = sample['images'][0]
    if image.mode != 'RGB':
        image = image.convert('RGB')
    image_inputs.append([image])

    # 2. Prepare the inputs
    model_inputs = processor(
        text=text_input,
        images=image_inputs,
        return_tensors="pt",
    ).to(device)

    # 3. Guardrailed Generation
    # We force it to generate at least 1 token so it can't immediately output an empty string
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=max_new_tokens,
        min_new_tokens=1,               # Prevents immediate EOS collapse
        do_sample=False,                # Greedy decoding is best for exact-match VQA
        repetition_penalty=1.1          # Slight penalty to prevent stuttering
    )

    # 4. Trim the generated ids to remove the input ids
    trimmed_generated_ids = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    # 5. Decode the output text
    output_text = processor.batch_decode(
        trimmed_generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )

    return output_text[0].strip()

In [ ]:
import gc
import time

def clear_memory():
    # Delete variables if they exist in the current global scope
    if 'inputs' in globals(): del globals()['inputs']
    if 'model' in globals(): del globals()['model']
    if 'processor' in globals(): del globals()['processor']
    if 'trainer' in globals(): del globals()['trainer']
    if 'peft_model' in globals(): del globals()['peft_model']
    if 'bnb_config' in globals(): del globals()['bnb_config']
    time.sleep(2)

    # Garbage collection and clearing CUDA memory
    gc.collect()
    time.sleep(2)
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    time.sleep(2)
    gc.collect()
    time.sleep(2)

    print(f"GPU allocated memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"GPU reserved memory: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

clear_memory()

In [ ]:
# from transformers import BitsAndBytesConfig

# # BitsAndBytesConfig int-4 config
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.bfloat16
# )

# # Load model and tokenizer
# model = Idefics3ForConditionalGeneration.from_pretrained(
#     model_id,
#     device_map="auto",
#     torch_dtype=torch.bfloat16,
#     quantization_config=bnb_config,
#     _attn_implementation="sdpa",
# )
# processor = AutoProcessor.from_pretrained(model_id)

In [ ]:
from torch import nn
class Idefics3SimpleMLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        input_size = config.vision_config.hidden_size
        output_size = config.text_config.hidden_size
        self.proj = nn.Linear(input_size, output_size, bias=False)

    def forward(self, x):
        return self.proj(x)

In [ ]:
from dataclasses import dataclass
from torch import nn
# -------------------------- SELECTOR START -------------------------- #
@dataclass
class ConnectorConfig:
    embed_dim: int = 1152      # 1152 * (2^2) for scale_factor=2
    num_selector_heads: int = 8
    selector_hidden_dim: int = 2048
    num_lqv: int = 32           # Number of condensed latents
    k_select: int = 360          # Number of anchor tokens to keep
    device: str = "cuda"

class DenseAttentionSelector(nn.Module):
    def __init__(self, embed_dim, num_heads, device):
        super().__init__()
        self.mha = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True, device=device)

    def forward(self, x, k):
        B, N, D = x.shape
        _, attn = self.mha(
            query=x,
            key=x,
            value=x,
            need_weights=True,
            average_attn_weights=False
        )  # (B, H, N, N)

        # 2. Compute Importance Scores (Density)
        # Sum across heads, then sum across keys to get per-token importance
        s1 = torch.sum(attn, dim=1)  # (B, N, N)
        s2 = torch.sum(s1, dim=1)    # (B, N)

        # 3. Sort indices by score (Descending)
        # This is the "Batch Secret": sorting keeps B intact and gives us a
        # map for both Top-K and 'The Rest'.
        _, top_idx = torch.topk(s2, k, dim=1)
        _, sorted_indices = torch.sort(s2, dim=1, descending=True) # (B, N)

        # 4. Partition Indices
        k_idx_raw = sorted_indices[:, :k]      # (B, k)
        q_idx_raw = sorted_indices[:, k:]      # (B, N-k)

        # 5. Expand indices for the Embedding Dimension (D)
        # We need (B, k, D) and (B, N-k, D) for torch.gather
        k_idx = k_idx_raw.unsqueeze(-1).expand(-1, -1, D)
        q_idx = q_idx_raw.unsqueeze(-1).expand(-1, -1, D)

        # 6. Gather the sets
        k_set = torch.gather(x, dim=1, index=k_idx)
        q_set = torch.gather(x, dim=1, index=q_idx)

        return k_set, q_set, top_idx

class QTokenCondenser(nn.Module):
    def __init__(self, embed_dim, num_heads, hidden_dim, num_lqv, device):
        super().__init__()
        self.lqv = nn.Parameter(torch.randn(1, num_lqv, embed_dim))

        self.mhca = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True, device=device)
        self.mha = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True, device=device)

        self.norm1 = nn.RMSNorm(embed_dim, device=device)
        self.norm2 = nn.RMSNorm(embed_dim, device=device)

        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim, device=device),
            nn.GELU(),
            nn.Linear(hidden_dim, embed_dim)
            )
    def forward(self, q):
        B, N, D  = q.shape
        lqv = self.lqv.expand(B, -1, -1)
        x, _ = self.mhca(query=lqv, key=q, value=q)
        x_norm = self.norm1(x)
        x = x_norm + self.mha(query=x_norm, key=x_norm, value=x_norm)[0]
        x_norm = self.norm2(x)
        x = x_norm + self.mlp(x_norm)
        return x
# -------------------------- SELECTOR END -------------------------- #

class Idefics3SelectorConnector(nn.Module):
    def __init__(self, config, custom_params):
        super().__init__()
        self.scale_factor = config.scale_factor
        self.modality_projection = Idefics3SimpleMLP(config)

        # -------------------------- SELECTOR START -------------------------- #
        self.condensor = QTokenCondenser(embed_dim=custom_params.embed_dim,
                                        num_heads=custom_params.num_selector_heads,
                                        hidden_dim = custom_params.selector_hidden_dim,
                                        num_lqv = custom_params.num_lqv,
                                        device = custom_params.device
                                        )
        self.k_select = custom_params.k_select
        self.selector = DenseAttentionSelector(embed_dim=custom_params.embed_dim,
                                              num_heads = custom_params.num_selector_heads,
                                              device=custom_params.device
                                              )

    def _init_weights(self):
        # 1. Zero out the final projection in the Condenser's MLP.
        # This ensures the condenser initially outputs values close to 0,
        # relying heavily on the residual connection.
        nn.init.zeros_(self.condensor.mlp[-1].weight)
        if self.condensor.mlp[-1].bias is not None:
            nn.init.zeros_(self.condensor.mlp[-1].bias)

        # 2. (Optional but recommended) Initialize the Selector's attention
        # so it roughly outputs a uniform distribution across all tokens at step 0,
        # preventing it from collapsing into picking the same bad patches.
        nn.init.xavier_uniform_(self.selector.mha.in_proj_weight)
        nn.init.zeros_(self.selector.mha.out_proj.weight)

    # -------------------------- SELECTOR END ---------------------------- #
    # def pixel_shuffle(self, x, scale_factor=2):
    #     bsz, seq, embed_dim = x.size()
    #     height = width = int(seq**0.5)
    #     x = x.view(bsz, height, width, embed_dim)
    #     x = x.view(bsz, height, int(width / scale_factor), embed_dim * scale_factor)
    #     x = x.permute(0, 2, 1, 3)
    #     x = x.reshape(bsz, int(width / scale_factor), int(height / scale_factor), embed_dim * (scale_factor**2))
    #     x = x.permute(0, 2, 1, 3)
    #     x = x.reshape(bsz, int(seq / (scale_factor**2)), embed_dim * (scale_factor**2))
    #     return x

    def forward(self, image_hidden_states):
        # image_hidden_states = self.pixel_shuffle(image_hidden_states, self.scale_factor)

        # -------------------------- SELECTOR START -------------------------- #
        k_set, q_set, top_idx = self.selector(image_hidden_states, self.k_select)
        if not self.training:
            self.last_top_idx = top_idx
        condensed_q = self.condensor(q_set)
        image_hidden_states = torch.cat([k_set, condensed_q], dim=1)
        # -------------------------- SELECTOR END -------------------------- #
        image_hidden_states = self.modality_projection(image_hidden_states)

        return image_hidden_states


In [ ]:
from transformers import Idefics3ForConditionalGeneration, BitsAndBytesConfig

# BitsAndBytesConfig int-4 config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Load model and tokenizer
model = Idefics3ForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    quantization_config=bnb_config,
    _attn_implementation="sdpa",
)

processor = AutoProcessor.from_pretrained(model_id)

# 2. Instantiate your custom config and connector
custom_params = ConnectorConfig(device=model.device)
custom_connector = Idefics3SelectorConnector(model.config, custom_params)

# 3. Apply your stable weight initialization
custom_connector._init_weights()

# 4. Perform the "surgery" - Replace the existing connector
# Make sure to move it to the correct device and dtype
model.model.connector = custom_connector.to(device=model.device, dtype=torch.bfloat16)

# 5. Freeze the backbone, Unfreeze the connector
for name, param in model.named_parameters():
    if "connector" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

# Verify trainable parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters in custom connector: {trainable}")

In [ ]:
# # Print all modules to find the "bridge" or "connector"
# for name, module in model.named_modules():
#     if "connector" in name or "model.modality_projection" in name:
#         print(f"Module: {name} | Type: {type(module)}")

In [ ]:
# from peft import LoraConfig, get_peft_model

# # Configure LoRA
# peft_config = LoraConfig(
#     r=8,
#     lora_alpha=8,
#     lora_dropout=0.1,
#     target_modules=['down_proj','o_proj','k_proj','q_proj','gate_proj','up_proj','v_proj'],
#     use_dora=True,
#     init_lora_weights="gaussian"
# )

# # Apply PEFT model adaptation
# peft_model = get_peft_model(model, peft_config)

# # Print trainable parameters
# peft_model.print_trainable_parameters()

In [ ]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir="smolvlm-instruct-trl-sft-ChartQA",
    num_train_epochs=10,
    per_device_train_batch_size=4, # You can likely bump this to 8 or 16 now!
    gradient_accumulation_steps=4,
    warmup_steps=50,
    learning_rate=1e-4,
    weight_decay=0.01,
    logging_steps=25,
    save_strategy="steps",
    save_steps=12,
    save_total_limit=2,
    optim="adamw_torch_fused",
    bf16=True,
    push_to_hub=True,
    report_to="none",
    max_length=None,

    # THE FIX: Explicitly disable gradient checkpointing
    gradient_checkpointing=False,
    dataloader_num_workers=4, # (Recommended) Keeps the A100 fed with data
)

In [ ]:
# import trackio

# trackio.init(
#     project="smolvlm-instruct-trl-sft-ChartQA",
#     name="smolvlm-instruct-trl-sft-ChartQA",
#     config=training_args,
#     space_id=training_args.output_dir + "-trackio"
# )

In [ ]:
trainable_breakdown = {}
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)
        # Group by the main component name to see where the bulk is
        component = name.split('.')[0] if '.' in name else name
        trainable_breakdown[component] = trainable_breakdown.get(component, 0) + param.numel()

print("Trainable Parameter Breakdown:")
for component, count in trainable_breakdown.items():
    print(f"{component}: {count:,}")

In [ ]:
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

# 1. The Trojan Horse PEFT Config
peft_config = LoraConfig(
    r=1,                     # The absolute minimum rank (computationally invisible)
    lora_alpha=1,            # Matches rank
    target_modules=["q_proj"], # Give PEFT exactly one layer to target so it doesn't crash
    modules_to_save=["connector"], # Unfreeze your custom architecture!
)

# # 2. Let SFTTrainer handle the wrapping automatically
# trainer = SFTTrainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_dataset,
#     eval_dataset=eval_dataset,
#     peft_config=peft_config, # Pass the config here!
#     processing_class=processor,
# )

# 3. Verify trainable parameters (Should be your ~239M connector + a tiny few KB for the r=1 adapter)
trainer.model.print_trainable_parameters()

# 4. Train!
# trainer.train()

In [ ]:
trainer.save_model(training_args.output_dir)

In [ ]:
clear_memory()

In [ ]:
import os
from safetensors.torch import load_file

# 1. Define your path
adapter_path = "/content/drive/MyDrive/smolvlm-instruct-trl-sft-ChartQA"
safetensors_path = os.path.join(adapter_path, "adapter_model.safetensors")

# 2. Load the raw dictionary of weights from the file
weights_dict = load_file(safetensors_path)

# 3. Clean the keys and filter out the dummy LoRA weights
clean_connector_weights = {}
for key, value in weights_dict.items():
    # Only grab the connector weights, ignore the dummy q_proj LoRA weights
    if "connector." in key:
        # Split at "connector." and take everything after it
        clean_key = key.split("connector.")[-1]
        clean_connector_weights[clean_key] = value

# 4. Identify the target module
target_module = model.model.connector
# If PEFT is still wrapping the module in the current notebook, unwrap it:
if hasattr(target_module, "original_module"):
    target_module = target_module.original_module

# 5. Load the clean weights directly into your custom architecture!
target_module.load_state_dict(clean_connector_weights, strict=True)

print(f"Successfully loaded {len(clean_connector_weights)} custom connector tensors!")

In [ ]:
i = 0
test_dataset[i]['images'][0]

In [ ]:
test_dataset[i]['messages'][1]['content'][1]['text']

In [ ]:
output = generate_text_from_sample_selector(model, processor, test_dataset[i])
output

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F

def visualize_isolated_patches(original_image, selected_indices, grid_size=(27, 27)):
    """
    Masks out non-selected patches, leaving only the high-density information
    visible on a black background.
    """
    # 1. Convert PIL image to NumPy (H, W, C)
    img_np = np.array(original_image)
    H, W, C = img_np.shape

    # 2. Create the binary grid mask (1 for selected, 0 for ignored)
    num_tokens = grid_size[0] * grid_size[1]
    mask = torch.zeros(num_tokens)
    mask[selected_indices.long()] = 1.0

    # Reshape to (1, 1, 27, 27) for interpolation
    mask = mask.view(1, 1, grid_size[0], grid_size[1])

    # 3. Upscale mask to the exact resolution of the image
    # 'nearest' ensures the edges of the patches stay sharp
    upscaled_mask = F.interpolate(mask, size=(H, W), mode='nearest').squeeze().numpy()

    # 4. Apply the mask across all 3 color channels
    # We multiply the image by the mask (mask is 0.0 or 1.0)
    masked_img = (img_np * upscaled_mask[:, :, np.newaxis]).astype(np.uint8)

    # 5. Display
    plt.figure(figsize=(12, 12))
    plt.imshow(masked_img)
    plt.axis('off')
    plt.title(f"Isolated Vision Inputs (Top-{len(selected_indices)} Tokens)")
    plt.show()

In [ ]:
# Run this inside your test loop, or on a single test sample
model.eval()
sample_idx = 3
sample = test_dataset[sample_idx]

# 1. Generate the answer (using the fixed prompt function!)
output = generate_text_from_sample(model, processor, sample)
print(f"Query: {sample['messages'][1]['content'][1]['text']}")
print(f"Model Answer: {output}")
print(f"True Label: {sample['messages'][2]['content'][0]['text']}")

# 2. Extract the selected indices
# We take [0] because we are processing batch size 1 during inference
selected_indices = model.model.connector.last_top_idx[0].cpu()

# 3. Visualize!
original_image = sample['images'][0]
visualize_isolated_patches(sample['images'][0], selected_indices)

In [ ]:
sample['images'][0]

In [ ]:
!pip install thop

In [ ]:
import torch
import time
from thop import profile # You may need to pip install thop

def benchmark_models(custom_model, processor, sample, iterations=10):
    device = "cuda"
    custom_model.eval()

    # 1. Prepare Inputs
    inputs = processor(text="<image>Answer:", images=[sample['images'][0]], return_tensors="pt").to(device)

    # 2. Benchmark Custom Model (Inference Time)
    torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.no_grad():
        for _ in range(iterations):
            _ = custom_model.generate(**inputs, max_new_tokens=5)
    torch.cuda.synchronize()
    custom_time = (time.perf_counter() - start) / iterations

    # 3. Benchmark Theoretical FLOPs (Connector Only)
    # Standard Idefics3 Projection: 10368 -> 2048
    std_flops = 10368 * 2048 * 81 # Dim_in * Dim_out * Tokens

    # Your Projection: 1152 -> 2048 (Simplified for comparison)
    # (Including your Selector Attention layers adds a bit more)
    custom_flops = 1152 * 2048 * 92

    print(f"--- Benchmark Results ---")
    print(f"Custom Model Inf Time: {custom_time:.4f}s")
    print(f"Standard Model Est. FLOPs (Proj): {std_flops:,}")
    print(f"Custom Model Est. FLOPs (Proj):   {custom_flops:,}")
    print(f"Efficiency Gain: {std_flops / custom_flops:.2f}x fewer FLOPs in projection")

# Run it
benchmark_models(model, processor, test_dataset[0])